In [1]:
# Image Filtering and Evaluation

# Step 1: Install required libraries
!pip install opencv-python matplotlib

# Step 2: Import libraries
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
import urllib.request

# Step 3: Upload an image or download a sample image
# Option 1: Upload from local
# uploaded = files.upload()
# image_path = list(uploaded.keys())[0]
# image = cv2.imread(image_path)

# Option 2: Download sample image
url = "https://raw.githubusercontent.com/opencv/opencv/master/samples/data/lena.jpg"
filename = "sample_image.jpg"
urllib.request.urlretrieve(url, filename)
image = cv2.imread(filename)

# Convert BGR to RGB for visualization
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Step 4: Apply Filters
# 1. Gaussian Blur
gaussian_blur = cv2.GaussianBlur(image_rgb, (15, 15), 0)

# 2. Median Blur
median_blur = cv2.medianBlur(image_rgb, 15)

# 3. Bilateral Filter
bilateral_filter = cv2.bilateralFilter(image_rgb, 15, 80, 80)

# 4. Sharpening Filter
kernel_sharpening = np.array([[-1,-1,-1],
                              [-1, 9,-1],
                              [-1,-1,-1]])
sharpened = cv2.filter2D(image_rgb, -1, kernel_sharpening)

# 5. Sobel Edge Detection
sobel_x = cv2.Sobel(image_rgb, cv2.CV_64F, 1, 0, ksize=5)
sobel_y = cv2.Sobel(image_rgb, cv2.CV_64F, 0, 1, ksize=5)
sobel_combined = cv2.magnitude(sobel_x, sobel_y)
# Normalize for display
sobel_norm = cv2.normalize(sobel_combined, None, 0, 255, cv2.NORM_MINMAX)
sobel_norm = np.uint8(sobel_norm)

# Step 5: Display images
titles = ['Original Image', 'Gaussian Blur', 'Median Blur',
          'Bilateral Filter', 'Sharpened Image', 'Sobel Edge Detection']
images = [image_rgb, gaussian_blur, median_blur, bilateral_filter, sharpened, sobel_norm]

plt.figure(figsize=(18, 10))
for i, (img, title) in enumerate(zip(images, titles)):
    plt.subplot(2, 3, i+1)
    if len(img.shape) == 3:
        plt.imshow(img)
    else:
        plt.imshow(img, cmap='gray')
    plt.title(title)
    plt.axis('off')
plt.tight_layout()
plt.show()

# Step 6: Quantitative Evaluation: MSE and PSNR
def mse(imageA, imageB):
    return np.mean((imageA.astype("float") - imageB.astype("float")) ** 2)

def psnr(imageA, imageB):
    m = mse(imageA, imageB)
    if m == 0:  # no difference
        return float('inf')
    max_pixel = 255.0
    return 10 * np.log10((max_pixel ** 2) / m)

for name, filtered in zip(['Gaussian', 'Median', 'Bilateral', 'Sharpened'],
                          [gaussian_blur, median_blur, bilateral_filter, sharpened]):
    print(f"{name} Filter -> MSE: {mse(image_rgb, filtered):.2f}, PSNR: {psnr(image_rgb, filtered):.2f} dB")

# Step 7: Save and download filtered images
save_names = ['gaussian_blur.jpg', 'median_blur.jpg', 'bilateral_filter.jpg', 'sharpened.jpg', 'sobel_edges.jpg']
save_images = [gaussian_blur, median_blur, bilateral_filter, sharpened, sobel_norm]

for name, img in zip(save_names, save_images):
    if len(img.shape) == 3:
        cv2.imwrite(name, cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
    else:
        cv2.imwrite(name, img)
    files.download(name)

Output hidden; open in https://colab.research.google.com to view.